Evaluation du modèle avant et après adaptation

Pour évaluer le modèle, on a deux critères :
- Classifications des actions. Chaque action donné par le système est labelisée via un "type" qui peut être
Enonciation;
Début;
Action.Valide;
Action.Invalide;
Action.Ambigue;
Action.Incohérente;
Exploration;
Fin. Cela nous fait 8 classes.

- la qualité du game master. Cela va mesurer si la réponse narrative respecte les règles établies du jeu.


Pour mesurer le avant et après adaptation, on va regarder l'indicateur de Parsing Rate qui va donner les fois où le modèle IA va générer un JSON parfaitement valide, l'accuracy qui va quantifier le nombre de prédictions correctes par rapport au total et le macro F1 qui va calculer la précision moyenne, beaucoup plus efficaace dans notre cas au vue des proportions de types et labels plus élevées chez Action.Valide que chez Fin par exemple.



Pour mesurer et évaluer le modèle, ce qui serait bien est de faire le test sans le fine tuning sur le jeu de test de 79 exemples du Dataset TEST. On récupère les scores reçus puis ensuite on fait passer le modèle fintuné directement sur le même dataset et on reprend les scores une nouvelles fois. A la fin, on regarder et on constate la différence entre le avant et après adaptation du modèle.


## 1. Installer les dépendances

In [ ]:
pip install -q -U transformers accelerate scikit-learn pandas matplotlib

## 2. Imports et configuration

In [ ]:
import json
import re
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TEST_FILE = Path("datasets/test.json")
OUTPUT_DIR = Path("evaluation_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_NEW_TOKENS = 220
DO_SAMPLE = False  # greedy decoding : génération déterministe

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Fichier de test :", TEST_FILE.resolve())

## 3. Charger le jeu de test

In [ ]:
if not TEST_FILE.exists():
    raise FileNotFoundError(
        f"{TEST_FILE} introuvable. Vérifie le chemin du fichier."
    )

with TEST_FILE.open("r", encoding="utf-8") as f:
    test_data = json.load(f)

test_examples = test_data["examples"]
print("Nombre d'exemples :", len(test_examples))
print("Clés du premier exemple :", test_examples[0].keys())
print(json.dumps(test_examples[0], ensure_ascii=False, indent=2)[:2500])

## 4. Normaliser les labels en huit classes

Le dataset contient des sous-types (`fin - victoire`, `action.valide - reformulation`) et des variantes orthographiques. On ramène les sous-types aux huit classes principales. Vérifie que cette règle correspond bien à la définition de vos classes avant de publier les résultats.

In [ ]:
CLASSES = [
    "énonciation",
    "début",
    "action.valide",
    "action.invalide",
    "action.ambigue",
    "action.incohérente",
    "exploration",
    "fin"
]

#### A voir si on établit vraiment 8 classes ou les déclinaisons de chacun avec Action.Valide - reformulation par exemple


def normalize_type(label):
    if label is None:
        return None
    s = str(label).strip().lower()
    s = re.sub(r"\s+", " ", s)

    # Normalisation des variantes et fautes connues.
    s = s.replace("énonciation", "enonciation")
    s = s.replace("début", "debut")
    s = s.replace("ambiguë", "ambigue")
    s = s.replace("incohérente", "incoherente")
    s = s.replace("inxohérente", "incoherente")

    if s.startswith("action.valide"):
        return "action.valide"
    if s.startswith("action.invalide"):
        return "action.invalide"
    if s.startswith("action.ambigue"):
        return "action.ambigue"
    if s.startswith("action.incoherente"):
        return "action.incohérente"
    if s.startswith("enonciation"):
        return "énonciation"
    if s.startswith("debut"):
        return "début"
    if s.startswith("exploration"):
        return "exploration"
    if s.startswith("fin"):
        return "fin"
    return None

raw_labels = [ex.get("type") for ex in test_examples]
y_true = [normalize_type(x) for x in raw_labels]
unknown = [(i, raw_labels[i]) for i, x in enumerate(y_true) if x is None]

print("Répartition normalisée :")
print(pd.Series(y_true).value_counts(dropna=False))
if unknown:
    print("Labels à traiter :", unknown)
    raise ValueError("Certains labels ne sont pas normalisés. Complète normalize_type.")

## 5. Visualiser la répartition des classes

In [ ]:
distribution = (
    pd.Series(y_true, name="classe")
    .value_counts()
    .reindex(CLASSES, fill_value=0)
    .rename_axis("classe")
    .reset_index(name="nombre")
)
distribution["pourcentage"] = distribution["nombre"] / len(y_true) * 100
display(distribution)

ax = distribution.plot(
    kind="bar", x="classe", y="nombre", legend=False,
    figsize=(10, 4), rot=35, title="Répartition des classes dans le test"
)
ax.set_ylabel("Nombre d'exemples")
plt.tight_layout()
plt.show()

## 6. Charger la baseline

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()
print("Modèle chargé :", MODEL_NAME)
print("Device :", model.device)

## 7. Construire les messages du modèle

On reprend l'historique présent dans chaque exemple. Le prompt demande une sortie JSON avec `type` et `response`. **Il faudra conserver le même prompt pour la baseline et le modèle fine-tuné**, sinon la comparaison sera moins fiable.

In [ ]:
OUTPUT_INSTRUCTION = (
    'Réponds uniquement avec un objet JSON valide, sans Markdown ni texte autour. '
    'Le JSON doit contenir "type" et "response". '
    '"type" doit être une seule de ces classes : '
    '["énonciation", "début", "action.valide", "action.invalide", '
    '"action.ambigue", "action.incohérente", "exploration", "fin"]. '
    '"response" est la réponse narrative du Game Master en français. '
    'Respecte les règles et l’historique du jeu.'
)

def build_messages(example):
    messages = [
        {"role": m["role"], "content": m["content"]}
        for m in example.get("messages", [])
        if m.get("role") in {"system", "user", "assistant"}
    ]
    system_indices = [i for i, m in enumerate(messages) if m["role"] == "system"]
    if system_indices:
        messages[system_indices[0]]["content"] += "\n\n" + OUTPUT_INSTRUCTION
    else:
        messages.insert(0, {
            "role": "system",
            "content": "Tu es le Game Master d'un jeu narratif.\n" + OUTPUT_INSTRUCTION
        })
    return messages

print(json.dumps(build_messages(test_examples[0]), ensure_ascii=False, indent=2)[:2500])

## 8. Fonction d'inférence

In [ ]:
def generate_response(example, model, tokenizer):
    messages = build_messages(example)
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

## 9. Parser et vérifier le JSON

Le Parsing Rate mesure ici la proportion de sorties qui sont des objets JSON syntaxiquement valides. Le taux de conformité au schéma vérifie en plus la présence d'un label connu et d'une réponse narrative non vide.

In [ ]:
def parse_generated_json(generated_text):
    try:
        obj = json.loads(generated_text)
    except json.JSONDecodeError as e:
        return None, False, False, f"JSON invalide : {e.msg}"

    if not isinstance(obj, dict):
        return None, False, False, "Le JSON n'est pas un objet"

    normalized = normalize_type(obj.get("type"))
    response = obj.get("response")
    schema_valid = (
        normalized in CLASSES
        and isinstance(response, str)
        and bool(response.strip())
    )
    if schema_valid:
        obj["type"] = normalized
        return obj, True, True, None
    return obj, True, False, "Schéma invalide ou champs manquants"

# Tests rapides du parseur
for sample in [
    '{"type":"action.valide","response":"Vous avancez."}',
    'texte avant {"type":"action.valide","response":"Vous avancez."}',
    '{"type":"inconnue","response":"Texte"}'
]:
    print(parse_generated_json(sample))

## 10. Lancer l'évaluation de la baseline

In [ ]:
def evaluate_model(examples, model, tokenizer, output_path, checkpoint_every=10):
    results = []
    output_path = Path(output_path)

    for i, example in enumerate(examples):
        start = time.time()
        generated = generate_response(example, model, tokenizer)
        parsed, json_valid, schema_valid, error = parse_generated_json(generated)

        expected = normalize_type(example.get("type"))
        predicted = parsed.get("type") if schema_valid and parsed else "erreur_de_format"
        narrative = parsed.get("response", "") if parsed else ""

        results.append({
            "example_index": i,
            "expected_type_raw": example.get("type"),
            "expected_type": expected,
            "generated_text": generated,
            "json_valid": json_valid,
            "schema_valid": schema_valid,
            "parse_error": error,
            "predicted_type": predicted,
            "narrative_response": narrative,
            "generation_seconds": round(time.time() - start, 3)
        })

        if (i + 1) % checkpoint_every == 0 or i == len(examples) - 1:
            with output_path.open("w", encoding="utf-8") as f:
                json.dump(results, f, ensure_ascii=False, indent=2)
            print(f"{i+1}/{len(examples)} exemples évalués")

    return results

baseline_results = evaluate_model(
    test_examples, model, tokenizer,
    OUTPUT_DIR / "predictions_baseline.json"
)

## 11. Calculer Parsing Rate, conformité, Accuracy et Macro F1

In [ ]:
def calculate_metrics(results, model_name="baseline"):
    df = pd.DataFrame(results)
    n = len(df)
    y_true_local = df["expected_type"].tolist()
    y_pred_local = df["predicted_type"].tolist()

    summary = {
        "modele": model_name,
        "nombre_exemples": n,
        "parsing_rate_%": 100 * df["json_valid"].mean(),
        "schema_rate_%": 100 * df["schema_valid"].mean(),
        "accuracy_%": 100 * accuracy_score(y_true_local, y_pred_local),
        "macro_f1_%": 100 * f1_score(
            y_true_local, y_pred_local, labels=CLASSES,
            average="macro", zero_division=0
        ),
        "temps_moyen_s": df["generation_seconds"].mean()
    }

    print(pd.Series(summary).to_string())
    print("\nRapport de classification :")
    print(classification_report(
        y_true_local, y_pred_local, labels=CLASSES,
        zero_division=0, digits=3
    ))

    report = pd.DataFrame(classification_report(
        y_true_local, y_pred_local, labels=CLASSES,
        output_dict=True, zero_division=0
    )).transpose()
    report.to_csv(
        OUTPUT_DIR / f"classification_report_{model_name}.csv",
        encoding="utf-8"
    )
    return summary, df, report

baseline_summary, baseline_df, baseline_report = calculate_metrics(
    baseline_results, "baseline"
)
display(pd.DataFrame([baseline_summary]))

## 12. Matrice de confusion

In [ ]:
def plot_confusion(df, title):
    cm = confusion_matrix(
        df["expected_type"], df["predicted_type"], labels=CLASSES
    )
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(cm, interpolation="nearest")
    ax.set_title(title)
    fig.colorbar(im, ax=ax)
    ax.set_xticks(range(len(CLASSES)))
    ax.set_yticks(range(len(CLASSES)))
    ax.set_xticklabels(CLASSES, rotation=45, ha="right")
    ax.set_yticklabels(CLASSES)
    ax.set_xlabel("Classe prédite")
    ax.set_ylabel("Classe attendue")
    threshold = cm.max() / 2 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > threshold else "black"
            )
    plt.tight_layout()
    plt.show()

plot_confusion(baseline_df, "Matrice de confusion — Baseline")

## 13. Préparer l'évaluation de la qualité narrative

La classification ne suffit pas à juger un Game Master. Pour chaque réponse, on évalue séparément :
- respect des règles ;
- cohérence avec l'historique ;
- pertinence par rapport à l'action du joueur ;
- qualité rédactionnelle et immersion.

Le fichier exporté peut être utilisé avec un LLM juge ou par un évaluateur humain. Si vous utilisez un LLM juge, gardez exactement la même grille pour les deux modèles et, si possible, faites vérifier un échantillon par un humain.

In [ ]:
JUDGE_RUBRIC = (
    "Tu évalues une réponse de Game Master de jeu de rôle fantasy. "
    "Attribue une note entière de 1 à 5 à chacun des critères : "
    "respect_regles, coherence, pertinence, immersion. "
    "1 = très mauvais / violation majeure ; 3 = acceptable ; 5 = excellent. "
    "Ne récompense pas simplement la longueur. Vérifie l'historique et les règles. "
    "Réponds uniquement en JSON avec les clés respect_regles, coherence, "
    "pertinence, immersion, justification."
)

def make_judge_item(example, result):
    return {
        "rubric": JUDGE_RUBRIC,
        "context": example.get("messages", []),
        "expected_type": result.get("expected_type"),
        "generated_response": result.get("narrative_response", "")
    }

judge_items = [
    {"example_index": i, **make_judge_item(test_examples[i], baseline_results[i])}
    for i in range(min(len(test_examples), len(baseline_results)))
]
with (OUTPUT_DIR / "judge_inputs_baseline.json").open("w", encoding="utf-8") as f:
    json.dump(judge_items, f, ensure_ascii=False, indent=2)

print("Exemples préparés pour le juge :", len(judge_items))
print(json.dumps(judge_items[0], ensure_ascii=False, indent=2)[:2000])

## 14. Grille de notation narrative à compléter

In [ ]:
rating_columns = ["respect_regles", "coherence", "pertinence", "immersion"]
human_ratings = pd.DataFrame({
    "example_index": range(len(baseline_results)),
    **{col: [np.nan] * len(baseline_results) for col in rating_columns},
    "justification": [""] * len(baseline_results)
})
ratings_file = OUTPUT_DIR / "ratings_baseline.csv"
human_ratings.to_csv(ratings_file, index=False, encoding="utf-8")
display(human_ratings.head())
print("Complète les notes de 1 à 5 dans :", ratings_file)

## 15. Résumer les notes narratives

In [ ]:
def summarize_ratings(csv_path):
    ratings = pd.read_csv(csv_path)
    criteria = ["respect_regles", "coherence", "pertinence", "immersion"]
    for col in criteria:
        ratings[col] = pd.to_numeric(ratings[col], errors="coerce")
        values = ratings[col].dropna()
        if not values.between(1, 5).all():
            raise ValueError(f"Les notes de {col} doivent être comprises entre 1 et 5.")
    print("Nombre de réponses avec au moins une note :",
          ratings[criteria].notna().any(axis=1).sum())
    display(ratings[criteria].mean().rename("moyenne / 5").to_frame())
    print("Moyenne globale /5 :", round(ratings[criteria].stack().mean(), 3))
    return ratings

# Après avoir complété le CSV :
# baseline_ratings = summarize_ratings(OUTPUT_DIR / "ratings_baseline.csv")

## 16. Évaluer le modèle fine-tuné après son entraînement

Quand le modèle adapté sera disponible, indique son dossier ci-dessous, charge-le puis lance l'évaluation sur **le même `test_examples`**. Ne lance pas cette cellule tant que le modèle n'existe pas.

In [ ]:
# À décommenter lorsque le fine-tuning sera terminé.
#
# FINETUNED_MODEL_DIR = "./qwen-finetuned"
# ft_tokenizer = AutoTokenizer.from_pretrained(FINETUNED_MODEL_DIR)
# ft_model = AutoModelForCausalLM.from_pretrained(
#     FINETUNED_MODEL_DIR, torch_dtype="auto", device_map="auto"
# )
# ft_model.eval()
#
# finetuned_results = evaluate_model(
#     test_examples, ft_model, ft_tokenizer,
#     OUTPUT_DIR / "predictions_finetuned.json"
# )
# finetuned_summary, finetuned_df, finetuned_report = calculate_metrics(
#     finetuned_results, "finetuned"
# )

## 17. Comparaison quantitative avant/après

In [ ]:
# À exécuter après l'évaluation du modèle fine-tuné.
#
# comparison = pd.DataFrame([baseline_summary, finetuned_summary]).set_index("modele")
# display(comparison)
#
# metrics = ["parsing_rate_%", "schema_rate_%", "accuracy_%", "macro_f1_%"]
# ax = comparison[metrics].T.plot(kind="bar", figsize=(10, 5))
# ax.set_ylabel("Pourcentage")
# ax.set_title("Comparaison baseline / fine-tuné")
# plt.xticks(rotation=20)
# plt.tight_layout()
# plt.show()
#
# print("Écart fine-tuné - baseline (points de pourcentage) :")
# display((comparison.loc["finetuned"] - comparison.loc["baseline"]).to_frame("écart"))

## 18. Comparaison humaine A/B à l'aveugle (optionnelle)

Présente les deux réponses pour le même contexte, dans un ordre aléatoire, sans révéler quel modèle les a générées. L'évaluateur choisit `baseline`, `finetuned` ou `egalite`, en se basant sur le respect des règles, la cohérence, la pertinence et l'immersion.

In [ ]:
# À exécuter après avoir créé baseline_results et finetuned_results.
#
# ab = pd.DataFrame({
#     "example_index": range(len(test_examples)),
#     "response_baseline": [r["narrative_response"] for r in baseline_results],
#     "response_finetuned": [r["narrative_response"] for r in finetuned_results],
#     "preference": [""] * len(test_examples),
#     "justification": [""] * len(test_examples)
# })
# ab.to_csv(OUTPUT_DIR / "blind_ab_comparison.csv", index=False, encoding="utf-8")
#
# Une fois le CSV complété :
# votes = pd.read_csv(OUTPUT_DIR / "blind_ab_comparison.csv")["preference"].value_counts()
# display(votes.to_frame("nombre"))
# display((100 * votes / votes.sum()).rename("pourcentage").to_frame())

## 19. Exporter les erreurs à analyser

In [ ]:
baseline_errors = baseline_df[
    (~baseline_df["json_valid"]) |
    (~baseline_df["schema_valid"]) |
    (baseline_df["expected_type"] != baseline_df["predicted_type"])
].copy()

baseline_errors.to_csv(
    OUTPUT_DIR / "baseline_errors.csv", index=False, encoding="utf-8"
)
pd.DataFrame([baseline_summary]).to_csv(
    OUTPUT_DIR / "summary_baseline.csv", index=False, encoding="utf-8"
)

print("Résultats sauvegardés dans :", OUTPUT_DIR.resolve())
print("Nombre d'exemples à inspecter :", len(baseline_errors))
display(baseline_errors[
    ["example_index", "expected_type", "predicted_type",
     "json_valid", "schema_valid", "generated_text"]
].head(10))